In [1]:
import pandas as pd
from pathlib import Path

In [2]:
OUTPUT_DIR = Path("../outputs/metrics")

user_metrics = pd.read_csv(
    OUTPUT_DIR / "user_metrics.csv"
)

control = user_metrics[
    user_metrics["group"] == "Control"
].copy()

treatment = user_metrics[
    user_metrics["group"] == "Treatment"
].copy()

control_rate = control["converted"].mean()
treatment_rate = treatment["converted"].mean()

absolute_uplift = treatment_rate - control_rate
relative_uplift = (
    (treatment_rate - control_rate) / control_rate
)

control_revenue = control["total_revenue"].sum()
treatment_revenue = treatment["total_revenue"].sum()

control_users = len(control)
treatment_users = len(treatment)

control_rpu = control_revenue / control_users
treatment_rpu = treatment_revenue / treatment_users

control_aov = (
    control_revenue / control["completed_orders"].sum()
)

treatment_aov = (
    treatment_revenue / treatment["completed_orders"].sum()
)

In [3]:
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

successes = [
    treatment["converted"].sum(),
    control["converted"].sum()
]

sample_sizes = [
    treatment_users,
    control_users
]

z_stat_one, p_value_one = proportions_ztest(
    count=successes,
    nobs=sample_sizes,
    alternative="larger"
)

from math import sqrt

diff = treatment_rate - control_rate

se_diff = sqrt(
    (treatment_rate * (1 - treatment_rate) / treatment_users)
    +
    (control_rate * (1 - control_rate) / control_users)
)

z_critical = stats.norm.ppf(0.975)

ci_lower = diff - z_critical * se_diff
ci_upper = diff + z_critical * se_diff

In [4]:
daily_users = 1_000_000

In [5]:
baseline_conversion = control_rate

In [6]:
new_conversion = treatment_rate

In [7]:
incremental_purchases = (
    daily_users *
    (new_conversion - baseline_conversion)
)

print(
    "Additional purchases/day:",
    incremental_purchases
)

Additional purchases/day: 23966.666666666693


In [8]:
incremental_revenue_daily = (
    incremental_purchases *
    treatment_aov
)

print(
    "Incremental daily revenue:",
    incremental_revenue_daily
)

Incremental daily revenue: 15482924.160390664


In [9]:
incremental_revenue_yearly = (
    incremental_revenue_daily *
    365
)

print(
    "Estimated annual incremental revenue:",
    incremental_revenue_yearly
)

Estimated annual incremental revenue: 5651267318.542592


In [10]:
treatment_profit_per_order = (
    treatment["total_profit"].sum()
    /
    treatment["completed_orders"].sum()
)

In [11]:
incremental_profit_daily = (
    incremental_purchases *
    treatment_profit_per_order
)

incremental_profit_yearly = (
    incremental_profit_daily *
    365
)

print(
    "Annual incremental profit:",
    incremental_profit_yearly
)

Annual incremental profit: 1514901366.3671923


In [12]:
implementation_cost = 500_000

In [13]:
roi = (
    incremental_profit_yearly
    -
    implementation_cost
) / implementation_cost

print(
    f"Estimated ROI: {roi:.2%}"
)

Estimated ROI: 302880.27%


In [14]:
statistically_significant = (
    p_value_one < 0.05
)

positive_effect = (
    absolute_uplift > 0
)

ci_positive = (
    ci_lower > 0
)

In [15]:
if (
    statistically_significant
    and
    positive_effect
    and
    ci_positive
):

    recommendation = (
        "ROLL OUT TREATMENT"
    )

elif (
    statistically_significant
    and
    absolute_uplift < 0
):

    recommendation = (
        "KEEP CONTROL"
    )

else:

    recommendation = (
        "DO NOT MAKE A FULL ROLLOUT DECISION. "
        "Collect more evidence."
    )

print(recommendation)

ROLL OUT TREATMENT


In [16]:
executive_summary = {
    "Control Conversion":
        control_rate,

    "Treatment Conversion":
        treatment_rate,

    "Absolute Uplift":
        absolute_uplift,

    "Relative Uplift":
        relative_uplift,

    "P Value":
        p_value_one,

    "95% CI Lower":
        ci_lower,

    "95% CI Upper":
        ci_upper,

    "Control AOV":
        control_aov,

    "Treatment AOV":
        treatment_aov,

    "Control RPU":
        control_rpu,

    "Treatment RPU":
        treatment_rpu,

    "Annual Incremental Revenue":
        incremental_revenue_yearly,

    "Annual Incremental Profit":
        incremental_profit_yearly,

    "Recommendation":
        recommendation
}

In [17]:
summary = pd.DataFrame(
    [executive_summary]
)

summary

,Control Conversion,Treatment Conversion,Absolute Uplift,Relative Uplift,P Value,95% CI Lower,95% CI Upper,Control AOV,Treatment AOV,Control RPU,Treatment RPU,Annual Incremental Revenue,Annual Incremental Profit,Recommendation
0,0.267317,0.291283,0.023967,0.089656,1.097289e-20,0.018892,0.029042,646.134613,646.019089,203.123185,226.741933,5.651267e+09,1.514901e+09,ROLL OUT TREATMENT


In [18]:
summary.to_csv(
    "../outputs/final_experiment_summary.csv",
    index=False
)